# Framework 4/4 — Actionable RCA Dashboard

> **Thesis:** Toward Actionable Root Cause Analysis: Integrating Causal Discovery and
> Counterfactual XAI for Predictive Maintenance
>
> **Prototype dataset:** NASA C-MAPSS FD001 · **Notebook 4 of 4** · 2026-07-30

The framework has nine modules. This notebook covers **Module 9 (dashboard & decision support)**.

All settings come from `pipeline_config.py` in the repo root. All logic lives in
`src/causalpdm/framework/`.

This notebook assembles every earlier stage into a single decision-support view: a
self-contained HTML file with no external dependencies, so it opens offline and can be projected,
emailed, or archived alongside the thesis.

## Setup

In [1]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT))

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)

import pipeline_config as cfg
from causalpdm.framework import artifacts

from causalpdm.framework import dashboard

prep = artifacts.load("prep")
feat = artifacts.load("feat")
disc = artifacts.load("disc")
valid = artifacts.load("valid") if artifacts.exists("valid") else None
scm_result = artifacts.load("scm")
rca_result = artifacts.load("rca")
interv = artifacts.load("interv")
cf = artifacts.load("cf")

print("All nine stages loaded.")
print(f"  engine analysed : {rca_result['unit']}")
print(f"  causal edges    : {disc['n_directed_edges']} discovered"
      + (f", {valid['n_validated']} validated" if valid else ""))
print(f"  best lever      : {interv['best_variable']}")
print(f"  counterfactual  : {'found' if cf['found'] else 'none achievable'}")

All nine stages loaded.
  engine analysed : 39
  causal edges    : 16 discovered, 8 validated
  best lever      : s_7
  counterfactual  : found


## What the dashboard has to communicate

A maintenance engineer needs four questions answered, in this order:

1. **Is this machine in trouble?** → health overview
2. **Why?** → causal graph and root cause ranking
3. **What should I do?** → intervention effects
4. **Would it have helped?** → counterfactual explanation

And then one thing that is usually missing from decision-support demos: **how much of this can I
trust?** The dashboard ends with an explicit scope panel separating results computed from data
(Modules 1–6) from model-based estimates that C-MAPSS cannot validate (Modules 7–8). A tool that
hides that distinction is a demo, not decision support.

## Generate the dashboard

In [2]:
output_file = REPO_ROOT / cfg.DASHBOARD_PATH
dashboard.render(
    prep, feat, disc, valid, scm_result, rca_result, interv, cf,
    out_path=output_file,
    dataset_label=f"NASA C-MAPSS {cfg.DATASET}",
)
print(f"Dashboard written to: {cfg.DASHBOARD_PATH}")
print(f"Size: {output_file.stat().st_size / 1024:.0f} KB (fully self-contained)")

Dashboard written to: reports/actionable_rca_dashboard.html
Size: 188 KB (fully self-contained)


## Preview it inline

In [3]:
from IPython.display import IFrame
IFrame(src=str(output_file.relative_to(Path.cwd())) if output_file.is_relative_to(Path.cwd())
       else f"../../{cfg.DASHBOARD_PATH}", width="100%", height=650)

If the preview above does not render in your Jupyter setup, open the file directly — it is a
normal HTML file:

```
reports/actionable_rca_dashboard.html
```

## Re-running the whole pipeline in one command

The four notebooks tell the story stage by stage. For a demo, or to regenerate everything after
changing a setting in `pipeline_config.py`, there is a single entry point:

```bash
python scripts/run_pipeline.py
```

It runs all nine modules and rewrites the dashboard. Cached stages are reused, so unless you
change a discovery setting it finishes in seconds.

## Framework summary

| Module | Purpose | Validated on C-MAPSS? |
|---|---|---|
| 1 — Preprocessing | Clean data, two representations | Yes |
| 2 — Features & health indicator | Degradation score, failure event definition | Yes |
| 3 — Causal discovery (PCMCI+) | Learn the causal graph | Runs, but no ground truth to score |
| 4 — Graph validation | Stability, sensitivity, cross-method, physics | Robustness only, not correctness |
| 5 — SCM fitting | Mechanisms + noise | Yes (fit quality measurable) |
| 6 — Root cause analysis | Separate causes from symptoms | Qualitatively |
| 7 — Intervention analysis | do-operations, effect ranking | **No — estimates only** |
| 8 — Counterfactual XAI | Abduction → action → prediction | **No — estimates only** |
| 9 — Dashboard | Decision support | N/A |

### Where this goes next

The two unvalidated modules are unvalidated because of the *dataset*, not the code. Every stage
interface is dataset-agnostic, so pointing the same pipeline at **causRCA** — which provides an
expert ground-truth causal graph and 100 labelled do-interventions — converts Modules 3, 4, 7 and
8 from demonstrations into measured results.

That is the thesis argument, and this prototype is the evidence that the pipeline works
end to end.